# Egyptian Real Estate EDA

### setup

In [1]:
import os
import sys

# 1. Set the IP to localhost to avoid the "Invalid Spark URL" error
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

# 2. Point to your Spark and Hadoop folders
os.environ['SPARK_HOME'] = r'C:\spark-4.1.1-bin-hadoop3\spark-4.1.1-bin-hadoop3'
os.environ['HADOOP_HOME'] = r'C:\Hadoop' 
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# 3. Add PySpark to your sys.path
sys.path.append(os.path.join(os.environ['SPARK_HOME'], 'python'))
sys.path.append(os.path.join(os.environ['SPARK_HOME'], 'python', 'lib', 'py4j-0.10.9.9-src.zip'))

# 4. Initialize Spark
from pyspark import SparkConf
from pyspark.sql import SparkSession
conf = SparkConf()
conf.set("spark.python.worker.timeout", "1200") # Set to 20 minutes
conf.set("spark.driver.memory", "4g")
conf.set("spark.executor.memory", "2g")
# If using a local-mode or single-node K8s, sometimes this helps:
conf.set("spark.driver.bindAddress", "127.0.0.1")
spark = SparkSession.builder \
    .appName("JupyterSparkTest") \
    .master("local[*]") \
    .config(conf=conf) \
    .getOrCreate()

sc = spark.sparkContext

### Load Dataset

In [2]:
import pandas as pd
import numpy as np

In [9]:
PATH  ="data/raw/propertyFinder.csv"
df_raw = pd.read_csv(PATH, encoding="utf-8-sig", low_memory=False)
print(f"Rows: {len(df_raw):,}  |  Columns: {df_raw.shape[1]}")
df_raw.head(3)

Rows: 39,713  |  Columns: 53


,listing_id,internal_id,category,listing_type,detail_url,property_type,offering_type,completion_status,title,price_egp,...,agent_is_super,agent_languages,broker_id,broker_name,broker_email,broker_phone,contact_phone,contact_whatsapp,contact_email,scraped_at
0,F7QB31CGWE509V2W7DF2GARB2C,56009081.0,buy,property,https://www.propertyfinder.eg/en/plp/buy/duple...,Duplex,Residential for Sale,completed,Garden Villa - Lake View Boutique - Prime Loca...,24500000.0,...,False,NaN,5758.0,Spade consultancy,mohmedg.sedik@gmail.com,1.001437e+09,2.012018e+11,2.022126e+10,pierre.Osama@spade-consultancy.com,2026-03-04T14:20:33.281007
1,K1JC3D6N57ED52N3VX1QQKHHXG,56247925.0,buy,property,https://www.propertyfinder.eg/en/plp/buy/apart...,Apartment,Residential for Sale,off_plan,For Sale: Finished Apartment+ ACs in Village West,5145000.0,...,False,NaN,492.0,Abrag Real Estate,snawara71@gmail.com,2.010012e+11,2.010072e+11,2.022126e+10,ranoushamer901@gmail.com,2026-03-04T14:20:33.281007
2,Q6GEB8T6PZTJGNNPWA5PX3JCWR,56253883.0,buy,property,https://www.propertyfinder.eg/en/plp/buy/apart...,Apartment,Residential for Sale,completed,UnderMarket Price for Apt. 250 RTM PrimeLocation,10800000.0,...,False,English | Arabic,5279.0,Premises,mahmouddessam@hotmail.com,2.010051e+11,2.010910e+11,2.022126e+10,ashrafsobhy99@hotmail.com,2026-03-04T14:20:33.281007


### Initial Inspection 

In [10]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 39713 entries, 0 to 39712
Data columns (total 53 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   listing_id           39712 non-null  str    
 1   internal_id          39712 non-null  float64
 2   category             39713 non-null  str    
 3   listing_type         39713 non-null  str    
 4   detail_url           39712 non-null  str    
 5   property_type        39712 non-null  str    
 6   offering_type        39712 non-null  str    
 7   completion_status    19763 non-null  str    
 8   title                39712 non-null  str    
 9   price_egp            39712 non-null  float64
 10  price_period         39712 non-null  str    
 11  price_currency       39713 non-null  str    
 12  location_full        39712 non-null  str    
 13  city                 39712 non-null  str    
 14  town                 39712 non-null  str    
 15  district             39366 non-null  str    
 1

In [11]:
df_raw.describe()

,internal_id,price_egp,lat,lon,area_value,images_count,rera,agent_id,broker_id,broker_phone,contact_phone,contact_whatsapp
count,3.971200e+04,3.971200e+04,39712.000000,39712.000000,39712.000000,39713.000000,0.0,39712.000000,39712.000000,3.951000e+04,3.971200e+04,3.971200e+04
mean,4.537948e+07,7.988803e+06,29.976419,31.318191,218.483229,12.275678,NaN,53288.360697,4144.885929,1.517036e+11,3.031897e+14,6.318752e+10
std,1.880498e+07,2.449279e+07,0.602531,0.857351,781.823867,4.823324,NaN,17909.357548,10005.156142,8.625258e+10,6.037916e+16,7.698019e+10
min,2.908155e+06,1.300000e+03,25.069366,25.476135,1.000000,0.000000,NaN,5500.000000,55.000000,1.000009e+09,2.010000e+11,2.022126e+10
25%,5.245428e+07,6.500000e+04,30.001993,31.015570,130.000000,10.000000,NaN,42365.000000,1614.000000,2.010001e+11,2.010330e+11,2.022126e+10
50%,5.476100e+07,6.000000e+05,30.025003,31.497156,175.000000,11.000000,NaN,57459.000000,4208.000000,2.010267e+11,2.011022e+11,2.022126e+10
75%,5.575865e+07,9.113125e+06,30.061470,31.546247,245.000000,14.000000,NaN,68818.000000,5657.000000,2.011170e+11,2.011456e+11,2.022126e+10
max,5.649342e+07,1.000000e+09,31.497467,34.896572,96600.000000,30.000000,NaN,73845.000000,179957.000000,2.015582e+11,1.203228e+19,2.015598e+11


In [12]:
missing = df_raw.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df_raw) * 100).round(1)

pd.DataFrame({"Missing Count": missing, "Percentage": missing_pct})

,Missing Count,Percentage
rera,39713,100.0
video_url,39511,99.5
agent_languages,30541,76.9
completion_status,19950,50.2
payment_method,19774,49.8
furnished,8385,21.1
subdistrict,6759,17.0
district,347,0.9
broker_email,203,0.5
broker_phone,203,0.5
